# Linear Algebra Refresher: Vectors, Dot Products, and Lines

The next lesson introduces the **perceptron**, the simplest neural network. Everything it computes comes down to one expression,

$$
\mathbf{w} \cdot \mathbf{x} + b,
$$

a **dot product** of a weight vector $\mathbf{w}$ with a data point $\mathbf{x}$, plus a number $b$. This short refresher reviews the linear algebra behind that expression, and the picture that goes with it: a line that splits the plane in two. You've seen most of this in a linear algebra course; the goal here is to connect it to the data and models in this course. If a section feels familiar, skim it and try its Ask.

You've already used this expression. Linear regression's prediction, $\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$, is a dot product plus a number. Neural-network people write the coefficients as $\mathbf{w}$ (for **weights**) and the intercept as $b$ (for **bias**), so this refresher does too.

### By the end, you'll be able to

1. Read a data point as a vector, both as a list of numbers and as an arrow, and add and scale vectors.
2. Compute a vector's length, and say why ridge's penalty is a squared length.
3. Compute a dot product two ways, and read its sign: same general direction, perpendicular, or opposite.
4. Describe the line $\mathbf{w} \cdot \mathbf{x} + b = 0$: it's perpendicular to $\mathbf{w}$, $b$ slides it, and $\mathbf{w}$ points toward the side where $\mathbf{w} \cdot \mathbf{x} + b > 0$.
5. Compute many dot products at once with a matrix, `X @ w`, and use their signs to sort points into two groups.

> **How to read the code.** As in the ridge and lasso notebook, cells marked **Code worth reading** are short numpy you should be able to write yourself. The demo cells only call functions from `ch_6_and_10_demos_helpers.py`: run them and use the sliders.

In [ ]:
import numpy as np

<a id="section-1"></a>

## 1 · Vectors: lists of numbers and arrows

A data point with $p$ features is a list of $p$ numbers, a **vector**: $\mathbf{x} = (x_1, x_2, \dots, x_p)$. With two features, you can draw it two ways: as a **point** in the plane at $(x_1, x_2)$, or as an **arrow** from the origin to that point. Both pictures will be useful.

The weights are a vector too, $\mathbf{w} = (w_1, \dots, w_p)$, with one weight per feature. So $\mathbf{w}$ and $\mathbf{x}$ live in the same space and can be drawn on the same axes.

Two operations:

- **Adding** two vectors adds matching entries: $(2, 1) + (-1, 3) = (1, 4)$. As arrows: put the second arrow's tail at the first arrow's tip; the sum runs from the start to the end.
- **Scaling** a vector multiplies every entry by the same number: $2 \times (1.5, 1) = (3, 2)$. As arrows: a positive number stretches or shrinks the arrow along the same direction, and a negative number also flips it around.

In [ ]:
from ch_6_and_10_demos_helpers import plot_vector_arithmetic

plot_vector_arithmetic();

Notice that the sum $\mathbf{w} + \mathbf{x}$ points **between** $\mathbf{w}$ and $\mathbf{x}$: adding $\mathbf{x}$ to $\mathbf{w}$ turns $\mathbf{w}$ toward $\mathbf{x}$. Keep that in mind for the next lesson.

**Code worth reading.** In numpy, `+`, `-`, and `*` on arrays work entry by entry, which is exactly vector addition and scaling.

In [ ]:
w = np.array([2, 1])
x = np.array([-1, 3])
print('w + x =', w + x)
print('2 * x =', 2 * x)
print('-x    =', -x)

**Ask:** With $\mathbf{w} = (2, 1)$ and $\mathbf{x} = (-1, 3)$, what is $\mathbf{w} - \mathbf{x}$? Draw it: how does it connect the tips of $\mathbf{x}$ and $\mathbf{w}$?

<details>
<summary>Answer</summary>

$\mathbf{w} - \mathbf{x} = (3, -2)$. It's the arrow from the tip of $\mathbf{x}$ to the tip of $\mathbf{w}$: start at $\mathbf{x}$, add $\mathbf{w} - \mathbf{x}$, and you land on $\mathbf{w}$.

</details>

<a id="section-2"></a>

## 2 · Length

The length of a vector, written $\lVert \mathbf{x} \rVert$, comes from the Pythagorean theorem:

$$
\lVert \mathbf{x} \rVert = \sqrt{x_1^2 + x_2^2 + \dots + x_p^2}
$$

For example, $\lVert (3, 4) \rVert = \sqrt{9 + 16} = 5$. Scaling multiplies the length: $\lVert 2\mathbf{x} \rVert = 2 \lVert \mathbf{x} \rVert$.

A **unit vector** has length 1. Dividing a vector by its length gives the unit vector pointing the same way: $(3, 4) / 5 = (0.6, 0.8)$. So every vector is a **direction** (its unit vector) times a **length**. That split comes back in the next lesson.

**You saw this today.** Ridge's penalty, $\beta_1^2 + \dots + \beta_p^2$, is the **squared length** of the coefficient vector $\beta$, so ridge charges for long coefficient vectors, and Demo M's circle is every $\beta$ whose length is at most $\sqrt{s}$. The lasso's $\lvert \beta_1 \rvert + \dots + \lvert \beta_p \rvert$ measures size a different way: the distance you'd walk from the origin if you could only move along the axes, like walking city blocks.

**Code worth reading.** `np.linalg.norm` computes the length.

In [ ]:
x = np.array([3, 4])
length = np.linalg.norm(x)
print('length:', length)
print('unit vector:', x / length)

**Ask:** What's the length of $(1, 1)$? Of $(1, 1, 1, 1)$?

<details>
<summary>Answer</summary>

$\sqrt{1 + 1} = \sqrt{2} \approx 1.41$, and $\sqrt{1 + 1 + 1 + 1} = 2$. The formula is the same in any number of dimensions; there are just more squares to add.

</details>

<a id="section-3"></a>

## 3 · The dot product

**The recipe.** Multiply matching entries, then add:

$$
\mathbf{w} \cdot \mathbf{x} = w_1 x_1 + w_2 x_2 + \dots + w_p x_p
$$

The result is a single number. For $\mathbf{w} = (2, 1)$ and $\mathbf{x} = (1, 3)$: $\mathbf{w} \cdot \mathbf{x} = 2 \cdot 1 + 1 \cdot 3 = 5$.

This is a **weighted sum** of the features, the same shape as a linear regression prediction: $\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p = \beta_0 + \beta \cdot \mathbf{x}$. Also, a vector dotted with itself is its squared length: $\mathbf{x} \cdot \mathbf{x} = \lVert \mathbf{x} \rVert^2$.

**The picture.** The same number is also

$$
\mathbf{w} \cdot \mathbf{x} = \lVert \mathbf{w} \rVert \, \lVert \mathbf{x} \rVert \cos\theta,
$$

where $\theta$ is the angle between the two arrows. In the example, $\lVert \mathbf{w} \rVert = \sqrt{5}$, $\lVert \mathbf{x} \rVert = \sqrt{10}$, and the angle is 45°, so this gives $\sqrt{50} \times 0.707 = 5$, the same answer.

You won't need trigonometry to use this. What matters most is the **sign**. Lengths are never negative, so the sign of $\mathbf{w} \cdot \mathbf{x}$ is the sign of $\cos\theta$:

| Angle between $\mathbf{w}$ and $\mathbf{x}$ | $\mathbf{w} \cdot \mathbf{x}$ | In words |
| --- | --- | --- |
| Under 90° | Positive | They point the same general way |
| Exactly 90° | Zero | They're **perpendicular** |
| Over 90° | Negative | They point in opposite general ways |

**The shadow.** Shine a light straight at the line through $\mathbf{w}$, so $\mathbf{x}$ casts a shadow on it. The shadow's signed length is $\mathbf{w} \cdot \mathbf{x} / \lVert \mathbf{w} \rVert$: positive if it falls ahead of the origin (on $\mathbf{w}$'s side), negative if behind. So **the dot product measures how far $\mathbf{x}$ reaches in $\mathbf{w}$'s direction**, scaled by the length of $\mathbf{w}$. In the example, the shadow is $5 / \sqrt{5} = \sqrt{5} \approx 2.24$.

<details>
<summary>Why do the recipe and the picture agree?</summary>

Draw the triangle with sides $\mathbf{w}$, $\mathbf{x}$, and $\mathbf{w} - \mathbf{x}$. The law of cosines says

$$
\lVert \mathbf{w} - \mathbf{x} \rVert^2 = \lVert \mathbf{w} \rVert^2 + \lVert \mathbf{x} \rVert^2 - 2 \lVert \mathbf{w} \rVert \lVert \mathbf{x} \rVert \cos\theta.
$$

Now expand the left side with the recipe instead. In two dimensions, $(w_1 - x_1)^2 + (w_2 - x_2)^2 = (w_1^2 + w_2^2) + (x_1^2 + x_2^2) - 2(w_1 x_1 + w_2 x_2)$, which is $\lVert \mathbf{w} \rVert^2 + \lVert \mathbf{x} \rVert^2 - 2\,\mathbf{w} \cdot \mathbf{x}$. Compare the two lines: $\mathbf{w} \cdot \mathbf{x} = \lVert \mathbf{w} \rVert \lVert \mathbf{x} \rVert \cos\theta$.

</details>

<a id="demo-n"></a>

### Demo N: The dot product as a shadow

Here $\mathbf{w}$ is fixed: length 2, pointing 30° above the $x_1$ axis, so $\mathbf{w} \approx (1.73, 1)$. The sliders set the direction and length of $\mathbf{x}$. The thick segment beside $\mathbf{w}$'s line is the shadow of $\mathbf{x}$: green when it falls ahead, pink when behind. The readout computes $\mathbf{w} \cdot \mathbf{x}$ both ways.

**Predict first:** At which two directions of $\mathbf{x}$ is the dot product exactly 0? Can making $\mathbf{x}$ longer ever change the sign?

In [ ]:
from ch_6_and_10_demos_helpers import dot_product_demo

dot_product_demo(size_percent=80)

<details>
<summary>What to notice</summary>

- **The recipe and the picture always agree.**
- **The dot product is 0 at 120° and 300°,** the two directions perpendicular to $\mathbf{w}$ (30° + 90° and 30° + 270°). There the shadow shrinks to a point.
- **The sign depends only on the direction.** Making $\mathbf{x}$ longer scales the dot product and the shadow, but never flips the sign.
- **It's largest when $\mathbf{x}$ points the same way as $\mathbf{w}$** (30°), and most negative when it points the opposite way (210°).

</details>

**Code worth reading.** `w @ x` computes the dot product. The angle comes from the picture's formula, solved for $\cos\theta$.

In [ ]:
w = np.array([2, 1])
x = np.array([1, 3])
print('w @ x   =', w @ x)
print('by hand =', 2 * 1 + 1 * 3)

cos_theta = (w @ x) / (np.linalg.norm(w) * np.linalg.norm(x))
print('angle   =', np.degrees(np.arccos(cos_theta)).round(1), 'degrees')

**Ask:** Without drawing, is the angle between $(1, 2)$ and $(3, -1)$ under, over, or exactly 90°? What about $(1, 2)$ and $(-2, 1)$?

<details>
<summary>Answer</summary>

$(1)(3) + (2)(-1) = 1$, which is positive, so the angle is **under 90°**.

$(1)(-2) + (2)(1) = 0$, so they're **perpendicular**: exactly 90°.

</details>

<a id="section-4"></a>

## 4 · A line from a dot product

Fix $\mathbf{w}$. Which points $\mathbf{x}$ have $\mathbf{w} \cdot \mathbf{x} = 0$? Exactly the ones perpendicular to $\mathbf{w}$, which form a **line through the origin, at a right angle to $\mathbf{w}$**. More generally, points with the same shadow on $\mathbf{w}$'s line have the same dot product. So each value of $\mathbf{w} \cdot \mathbf{x}$ picks out a line perpendicular to $\mathbf{w}$, and different values give parallel lines.

Adding a number $b$ shifts the line. The points with

$$
\mathbf{w} \cdot \mathbf{x} + b = 0
$$

form a line perpendicular to $\mathbf{w}$, moved away from the origin. For example, with $\mathbf{w} = (1, 1)$ and $b = -2$, it's $x_1 + x_2 - 2 = 0$: the line through $(2, 0)$ and $(0, 2)$.

Call $\mathbf{w} \cdot \mathbf{x} + b$ the **score** of $\mathbf{x}$. The line splits the plane into two sides:

- On the side $\mathbf{w}$ points toward, the score is **positive**.
- On the other side, it's **negative**.

The score's sign says which side a point is on, and its size says how far away it is: the distance from the line is $\lvert \mathbf{w} \cdot \mathbf{x} + b \rvert / \lVert \mathbf{w} \rVert$. In the example, $(3, 0)$ has score $3 + 0 - 2 = 1$, so it's on the positive side, at distance $1 / \sqrt{2} \approx 0.71$ from the line.

**This is the perceptron's picture.** A perceptron sorts points by the sign of their score: one class on each side of the line. The next lesson is about how it finds $\mathbf{w}$ and $b$ from data.

<a id="demo-o"></a>

### Demo O: A line from $\mathbf{w}$ and $b$

The sliders set the direction of $\mathbf{w}$, its length, and $b$. Orange shading marks the positive side and blue the negative side. The black line is the boundary, where the score is 0, and the dashed lines are where the score is $\pm 1, \pm 2, \dots$ The arrow is $\mathbf{w}$, drawn from the point on the line closest to the origin. The readout gives the scores of three fixed points, A, B, and C.

Move one slider at a time:

1. **Direction of $\mathbf{w}$.**
2. **$b$.**
3. **Length of $\mathbf{w}$,** first with $b = 0$, then with $b \neq 0$.

**Predict first:** Which slider slides the line without turning it? With $b = 0$, which slider changes the scores without moving the line at all?

In [ ]:
from ch_6_and_10_demos_helpers import line_from_dot_product_demo

line_from_dot_product_demo(size_percent=80)

<details>
<summary>What to notice</summary>

- **The direction of $\mathbf{w}$ turns the line.** The line stays perpendicular to $\mathbf{w}$, and $\mathbf{w}$ always points into the orange (positive) side.
- **$b$ slides the line** without turning it. A bigger $b$ raises every score, so the positive side grows and the line backs away from the direction $\mathbf{w}$ points. With $b = 0$, the line passes through the origin.
- **The length of $\mathbf{w}$ changes the scores.** With $b = 0$, a longer $\mathbf{w}$ leaves the line where it is but packs the dashed lines closer together (they're $1 / \lVert \mathbf{w} \rVert$ apart), so every point's score grows. With $b \neq 0$, a longer $\mathbf{w}$ also pulls the line toward the origin, because its distance from the origin is $\lvert b \rvert / \lVert \mathbf{w} \rVert$.
- **A point's score changes sign exactly when the line crosses it.** Watch A, B, and C in the readout.
- **Different $\mathbf{w}$ and $b$ can give the same line.** Multiplying both by the same positive number leaves the line exactly where it is and scales every score. The sign alone can't tell them apart; the next lesson puts the size of the score to use.

</details>

**More features.** With three features, $\mathbf{w} \cdot \mathbf{x} + b = 0$ is a plane in 3D. With $p$ features, it's a flat $(p - 1)$-dimensional surface called a **hyperplane**. It's still perpendicular to $\mathbf{w}$, and it still splits the space into two sides. Our pictures stay in 2D, and the formulas don't change.

**Ask:** For $\mathbf{w} = (1, 1)$ and $b = -2$: (a) Is the origin on the positive or the negative side? (b) What's the score of $(2, 2)$, and how far is it from the line? (c) Which of these give the same line: $\mathbf{w} = (2, 2), b = -4$; $\mathbf{w} = (-1, -1), b = 2$; $\mathbf{w} = (1, 1), b = -4$?

<details>
<summary>Answer</summary>

(a) **Negative.** Its score is $0 + 0 - 2 = -2$.

(b) **Score $2 + 2 - 2 = 2$, distance $2 / \sqrt{2} = \sqrt{2} \approx 1.41$.**

(c) **The first two.** Each is the original equation multiplied by a number (2, then $-1$), so the same points satisfy it. But multiplying by $-1$ flips $\mathbf{w}$ around, which swaps the positive and negative sides. The third, $x_1 + x_2 = 4$, is a parallel line farther from the origin.

</details>

<a id="section-5"></a>

## 5 · Many dot products at once: a matrix times a vector

A data set with $n$ points and $p$ features is a matrix $X$ with $n$ rows and $p$ columns, one row per point. Multiplying $X$ by $\mathbf{w}$ computes the dot product of **every row** with $\mathbf{w}$ at once:

$$
X\mathbf{w} = \begin{pmatrix} \text{row}_1 \cdot \mathbf{w} \\ \text{row}_2 \cdot \mathbf{w} \\ \vdots \\ \text{row}_n \cdot \mathbf{w} \end{pmatrix}
$$

The shapes: an $n \times p$ matrix times a vector with $p$ entries gives a vector with $n$ entries. The $p$'s must match, since each row needs one weight per feature.

**Code worth reading.** Five points, scored against Section 4's line ($\mathbf{w} = (1, 1)$, $b = -2$). `X @ w + b` gives all five scores, and `np.sign` reads off each point's side.

In [ ]:
X = np.array([[3, 0],
              [0, 0],
              [1, 1],
              [2, 2],
              [-1, 4]])
w = np.array([1, 1])
b = -2

scores = X @ w + b
print('scores:', scores)
print('sides: ', np.sign(scores))

That's a complete classifier: score each point, then read its class from the sign. The point $(1, 1)$ sits exactly on the line, with a score of 0. The next lesson is about how a perceptron finds $\mathbf{w}$ and $b$ from labeled data.

**Many $\mathbf{w}$'s at once.** Stack several weight vectors as the rows of a matrix $W$. Then $W\mathbf{x}$ gives each one's score for the same point $\mathbf{x}$. A layer of a neural network is exactly this: many $\mathbf{w}$'s, each scoring the same input. We'll come back to it with networks.

**Ask:** $X$ has 100 rows and 3 columns. (a) How many entries must $\mathbf{w}$ have for `X @ w` to work? (b) How many entries does `X @ w` have? (c) What does its 7th entry mean?

<details>
<summary>Answer</summary>

(a) **3,** one weight per feature. (numpy raises a `ValueError` about mismatched shapes otherwise.)

(b) **100,** one per point.

(c) **The dot product of the 7th point (row 7 of $X$) with $\mathbf{w}$:** that point's score, before adding $b$.

</details>

<a id="section-6"></a>

## 6 · Summary, and next time

- A **vector** is a list of numbers. With two entries, it's a point or an arrow. Vectors add entry by entry (tip to tail), and scaling stretches, shrinks, or flips them.
- **Length:** $\lVert \mathbf{x} \rVert$ is the square root of the sum of squares. Every vector is a direction times a length. Ridge's penalty is the squared length of $\beta$.
- **Dot product:** multiply matching entries and add. It equals $\lVert \mathbf{w} \rVert \lVert \mathbf{x} \rVert \cos\theta$, so its sign says whether two vectors point the same general way (positive), are perpendicular (zero), or point opposite ways (negative). $\mathbf{w} \cdot \mathbf{x} / \lVert \mathbf{w} \rVert$ is the shadow of $\mathbf{x}$ on $\mathbf{w}$'s direction.
- **A line:** $\mathbf{w} \cdot \mathbf{x} + b = 0$ is perpendicular to $\mathbf{w}$, $b$ slides it, and $\mathbf{w}$ points to the side with positive scores. A point's distance from the line is $\lvert \text{score} \rvert / \lVert \mathbf{w} \rVert$.
- **A matrix times a vector:** `X @ w + b` scores every point at once, and the signs sort the points into two sides.
- **Next:** the perceptron, a line $\mathbf{w} \cdot \mathbf{x} + b = 0$ that learns from data where to sit.

# Practice Questions

These questions cover vectors, length, the dot product, and the line $\mathbf{w} \cdot \mathbf{x} + b = 0$. Try each one on paper before you open its answer.

### Question 1: A dot product

For $\mathbf{w} = (3, -1)$ and $\mathbf{x} = (2, 4)$: (a) compute $\mathbf{w} \cdot \mathbf{x}$. (b) Is the angle between them under, over, or exactly 90°? (See [Section 3](#section-3).)

<details>
<summary>Answer</summary>

(a) $(3)(2) + (-1)(4) = 2$.

(b) **Under 90°,** because the dot product is positive.

</details>

### Question 2: Perpendicular

Which of $(-1, 2)$, $(-2, -1)$, $(1, -2)$, and $(4, 2)$ are perpendicular to $(2, 1)$? For the others, do they point the same general way as $(2, 1)$, or the opposite way? (See [Section 3](#section-3).)

<details>
<summary>Answer</summary>

**$(-1, 2)$ and $(1, -2)$** are perpendicular: both dot products are 0.

$(4, 2)$ has dot product 10, so it points the same way (in fact, exactly the same direction: it's $2 \times (2, 1)$). $(-2, -1)$ has dot product $-5$, so it points the opposite way.

</details>

### Question 3: Length

For $\mathbf{x} = (6, 8)$: (a) What's $\lVert \mathbf{x} \rVert$? (b) What's the unit vector in the direction of $\mathbf{x}$? (c) What's the length of $3\mathbf{x}$? (See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

(a) $\sqrt{36 + 64} = 10$.

(b) $(6, 8) / 10 = (0.6, 0.8)$.

(c) $3 \times 10 = 30$.

</details>

### Question 4: Which side?

Take $\mathbf{w} = (2, -1)$ and $b = 1$. (a) Find the scores of $(1, 1)$ and $(0, 3)$, and say which side of the line each is on. (b) How far is $(1, 1)$ from the line? (c) Do $\mathbf{w} = (4, -2)$ and $b = 2$ give the same line? The same scores? (See [Section 4](#section-4).)

<details>
<summary>Answer</summary>

(a) $(1, 1)$: $2 - 1 + 1 = 2$, **positive** side. $(0, 3)$: $0 - 3 + 1 = -2$, **negative** side.

(b) $2 / \lVert (2, -1) \rVert = 2 / \sqrt{5} \approx 0.89$.

(c) **The same line, but every score doubles** ($4$ and $-4$). Both $\mathbf{w}$ and $b$ were multiplied by 2.

</details>

### Question 5: Shapes

$X$ has 50 rows and 4 columns, and `X @ w` works. (a) How many entries does $\mathbf{w}$ have? (b) How many entries does `X @ w` have, and what does each one mean? (See [Section 5](#section-5).)

<details>
<summary>Answer</summary>

(a) **4,** one per feature.

(b) **50,** one per point. Entry $i$ is the dot product of row $i$ (point $i$) with $\mathbf{w}$.

</details>

### Question 6: Back to ridge

A model has coefficients $\beta = (3, 4)$. (a) What is ridge's penalty $\beta_1^2 + \beta_2^2$, and what does it measure? (b) What is the lasso's penalty $\lvert \beta_1 \rvert + \lvert \beta_2 \rvert$? (Leave out $\lambda$.) (See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

(a) $9 + 16 = 25$: the **squared length** of $\beta$ (its length is 5).

(b) $3 + 4 = 7$: the city-block distance from the origin to $\beta$.

</details>